# Donor Response Prediction and Used Clothes Image Clustering
## End-to-End Machine Learning Portfolio Project

This notebook applies supervised machine learning to predict donor response and unsupervised image clustering to support used-clothes condition review.

#### Load the dependencies

The first step is to import the libraries required for data preparation, supervised learning, model evaluation, image feature extraction, dimensionality reduction, clustering and visualisation.

In [ ]:
# Load the libraries needed
import os
import shutil
import textwrap
from pathlib import Path
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    davies_bouldin_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

RANDOM_STATE = 42

# Resolve project paths whether the notebook is launched from the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

## Part 1. Analysis of Structured Data: Predict Donor Donation Behaviour

The structured analysis predicts whether a solicited donor will donate. Logistic Regression is used as an interpretable baseline, while Random Forest is used to capture non-linear relationships and interactions between donor characteristics.

#### Load the donor data

The `DONOR.csv` file is read from `data/raw/`. Raw data is kept outside version control so that the public repository remains suitable for a portfolio.

In [ ]:
# Load the structured donor data
donor_data = pd.read_csv(DATA_DIR / "DONOR.csv")

print(f"Number of rows: {donor_data.shape[0]}")
print(f"Number of columns: {donor_data.shape[1]}")
donor_data.head()

#### Inspect data quality and the target distribution

The following checks identify data types, missing values, duplicate records and class imbalance before model development.

In [ ]:
# Inspect the dataset
display(donor_data.dtypes.to_frame("Data Type"))

missing_summary = pd.DataFrame({
    "Missing Values": donor_data.isna().sum(),
    "Missing Percentage": donor_data.isna().mean().mul(100).round(2)
}).sort_values("Missing Values", ascending=False)
display(missing_summary)

print(f"Number of duplicated rows: {donor_data.duplicated().sum()}")
display(donor_data["DONATE"].value_counts().rename_axis("DONATE").to_frame("Count"))
display(
    donor_data["DONATE"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename_axis("DONATE")
    .to_frame("Percentage")
)

#### Define the target and predictor variables

`DONATE` is converted to a binary target where No = 0 and Yes = 1. `AMOUNT` is removed because it records the outcome of the current solicitation and would leak the target into the models. `ID` is also excluded because it is an identifier rather than a donor characteristic.

In [ ]:
# Convert the target variable and remove non-predictive or leakage variables
y = donor_data["DONATE"].map({"No": 0, "Yes": 1})
X = donor_data.drop(columns=["DONATE", "AMOUNT", "ID"])

if y.isna().any():
    raise ValueError("DONATE contains values other than 'No' and 'Yes'.")

print("Predictor variables:")
print(X.columns.tolist())
print("\nTarget coding: No = 0, Yes = 1")

#### Create training and testing datasets

The data is divided into 80% training data and 20% testing data. Stratification preserves the Yes/No class proportions in both datasets. All imputers, scalers and encoders are fitted only on the training data through model pipelines.

In [ ]:
# Split the data while preserving the target class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Rows": [len(X_train), len(X_test)],
    "Donation Rate": [y_train.mean(), y_test.mean()]
})
split_summary["Donation Rate"] = split_summary["Donation Rate"].round(4)
split_summary

#### Define the data pre-processing steps

* `AGE` and `WEALTH_RATING` are imputed using training-set medians.
* `SES` is imputed using the training-set mode and remains numeric without One-Hot Encoding.
* Missing `URBANICITY` values are assigned to a new `MISSING` category before One-Hot Encoding.
* All remaining categorical predictors are imputed if necessary and One-Hot Encoded.
* Numeric predictors are standardised so that Logistic Regression is not dominated by variables with larger scales.

In [ ]:
# Identify variables requiring different preprocessing steps
median_numeric_features = [
    "AGE",
    "WEALTH_RATING",
    "FREQUENCY",
    "RESPONSE_PROP",
    "LAST_AMOUNT",
    "MONTHS_SINCE_LAST_DONATION"
]
ses_feature = ["SES"]
urbanicity_feature = ["URBANICITY"]
other_categorical_features = ["HOME_OWNER", "GENDER", "STATUS"]

# Median imputation and scaling for numeric variables
median_numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Mode imputation for SES; SES is not One-Hot Encoded
ses_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler())
])

# Create an explicit MISSING category for URBANICITY before One-Hot Encoding
urbanicity_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# One-Hot Encode all remaining categorical variables
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combine all preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", median_numeric_pipeline, median_numeric_features),
        ("ses", ses_pipeline, ses_feature),
        ("urbanicity", urbanicity_pipeline, urbanicity_feature),
        ("categorical", categorical_pipeline, other_categorical_features)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Preprocessing pipeline created successfully.")

#### Train the two machine learning models

Logistic Regression estimates a linear probability boundary and provides interpretable coefficients. Random Forest combines many decision trees to model non-linear patterns and predictor interactions. Balanced class weights are used because only approximately one quarter of the records belong to the donation class.

In [ ]:
# Create model pipelines so preprocessing is fitted only on the training data
models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}

# Fit both models
for model_name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{model_name} training completed.")

#### Test and evaluate the models

The models are compared using Accuracy, Precision, Recall, F1 Score and ROC-AUC. Because the target is imbalanced, Recall, F1 Score and ROC-AUC provide important information beyond overall Accuracy.

In [ ]:
# Generate test predictions and calculate evaluation metrics
evaluation_rows = []
model_predictions = {}

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    y_probability = model.predict_proba(X_test)[:, 1]
    model_predictions[model_name] = {
        "prediction": y_pred,
        "probability": y_probability
    }

    evaluation_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_probability)
    })

    print(f"\n{model_name} Classification Report")
    print(classification_report(
        y_test,
        y_pred,
        target_names=["No Donation", "Donation"],
        zero_division=0
    ))

model_comparison = pd.DataFrame(evaluation_rows).set_index("Model")
display(model_comparison.round(4))

best_model_name = model_comparison["ROC-AUC"].idxmax()
print(f"Best model based on test ROC-AUC: {best_model_name}")

In [ ]:
# Display confusion matrices for both models
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for axis, (model_name, results) in zip(axes, model_predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        results["prediction"],
        display_labels=["No", "Yes"],
        cmap="Blues",
        colorbar=False,
        ax=axis
    )
    axis.set_title(f"{model_name} Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
# Plot ROC curves for model comparison
plt.figure(figsize=(8, 6))

for model_name, results in model_predictions.items():
    false_positive_rate, true_positive_rate, _ = roc_curve(
        y_test,
        results["probability"]
    )
    auc_value = roc_auc_score(y_test, results["probability"])
    plt.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        label=f"{model_name} (AUC = {auc_value:.3f})"
    )

plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#### Examine the most influential predictors

Logistic Regression coefficients show the direction and strength of associations after preprocessing. Random Forest feature importance shows which transformed predictors contribute most to its decisions, but does not indicate the direction of each relationship.

In [ ]:
# Display the largest Logistic Regression coefficients by absolute magnitude
logistic_pipeline = models["Logistic Regression"]
logistic_feature_names = logistic_pipeline.named_steps["preprocessor"].get_feature_names_out()
logistic_coefficients = logistic_pipeline.named_steps["classifier"].coef_[0]

coefficient_table = pd.DataFrame({
    "Feature": logistic_feature_names,
    "Coefficient": logistic_coefficients
})
top_coefficient_indices = (
    coefficient_table["Coefficient"]
    .abs()
    .sort_values(ascending=False)
    .head(15)
    .index
)
top_coefficients = coefficient_table.loc[top_coefficient_indices].sort_values("Coefficient")

plt.figure(figsize=(10, 7))
bar_colours = np.where(top_coefficients["Coefficient"] >= 0, "#2E86AB", "#D1495B")
plt.barh(top_coefficients["Feature"], top_coefficients["Coefficient"], color=bar_colours)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient")
plt.title("Top Logistic Regression Coefficients")
plt.tight_layout()
plt.show()

display(top_coefficients.sort_values("Coefficient", ascending=False).reset_index(drop=True))

In [ ]:
# Display the most important Random Forest predictors
forest_pipeline = models["Random Forest"]
forest_feature_names = forest_pipeline.named_steps["preprocessor"].get_feature_names_out()
forest_importances = forest_pipeline.named_steps["classifier"].feature_importances_

importance_table = (
    pd.DataFrame({
        "Feature": forest_feature_names,
        "Importance": forest_importances
    })
    .sort_values("Importance", ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 7))
plt.barh(
    importance_table["Feature"][::-1],
    importance_table["Importance"][::-1],
    color="#2E86AB"
)
plt.xlabel("Feature Importance")
plt.title("Top Random Forest Feature Importances")
plt.tight_layout()
plt.show()

display(importance_table.reset_index(drop=True))

## Part 2. Analysis of Unstructured Data: Detecting Damaged Clothes

The image data is unlabelled, so K-Means clustering is used to group visually similar clothes. A pre-trained VGG16 convolutional neural network transforms each image into numerical features, PCA reduces feature dimensionality, and the Davies-Bouldin Index supports the selection of the number of clusters.

#### Unzip the unlabelled image data

The image archive is extracted into `data/raw/`. The extraction step is skipped if the target folder already exists, allowing the notebook to be run more than once.

In [ ]:
# Unzip used_clothes_images.zip
archive_path = DATA_DIR / "used_clothes_images.zip"
image_folder = DATA_DIR / "used_clothes_images"

if not os.path.isdir(image_folder):
    shutil.unpack_archive(archive_path, DATA_DIR)
    print(f"Extracted {archive_path}.")
else:
    print(f"Folder '{image_folder}' already exists; extraction was skipped.")

#### Define a function to load images from a folder

Each image is loaded, converted from OpenCV BGR format to RGB, and resized to 150 x 150 pixels. Sorting the filenames ensures reproducible correspondence between images and exported cluster labels.

In [ ]:
# Load and prepare images from the folder
def load_images_from_folder(folder, image_size=(150, 150)):
    images = []
    filenames = []
    supported_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    for filename in sorted(os.listdir(folder)):
        if not filename.lower().endswith(supported_extensions):
            continue

        file_path = os.path.join(folder, filename)
        image = cv2.imread(file_path)

        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, image_size)
            images.append(image)
            filenames.append(filename)

    if not images:
        raise ValueError(f"No readable images were found in '{folder}'.")

    return np.array(images), np.array(filenames)


In [ ]:
# Load the used-clothes images
images, filenames = load_images_from_folder(image_folder)

print(f"Number of images loaded: {len(images)}")
print(f"Image array shape: {images.shape}")

#### Extract image features using the pre-trained VGG16 model

VGG16 was pre-trained on ImageNet and can extract general visual patterns such as edges, textures and shapes. The final classification layers are excluded because the objective is to obtain reusable image features rather than ImageNet class predictions.

In [ ]:
# Extract image features using the pre-trained VGG16 model
def extract_features(images, batch_size=32):
    feature_extractor = VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=(150, 150, 3)
    )
    feature_extractor.trainable = False

    prepared_images = preprocess_input(images.astype(np.float32).copy())
    features = feature_extractor.predict(
        prepared_images,
        batch_size=batch_size,
        verbose=1
    )
    return features


In [ ]:
# Apply VGG16 feature extraction
cnn_features = extract_features(images)
print(f"CNN feature shape: {cnn_features.shape}")

#### Flatten the features and apply PCA

The multidimensional VGG16 output is flattened into one vector per image. PCA then reduces the features to 50 components, which can remove redundant variation and make clustering more computationally efficient. The first two components are retained separately for visualisation.

In [ ]:
# Flatten the CNN features
flattened_features = cnn_features.reshape(cnn_features.shape[0], -1)

# Reduce the feature dimensions using PCA
n_components = min(50, flattened_features.shape[0] - 1, flattened_features.shape[1])
pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
reduced_features = pca.fit_transform(flattened_features)

print(f"Flattened feature shape: {flattened_features.shape}")
print(f"PCA feature shape: {reduced_features.shape}")
print(f"Variance explained by {n_components} components: {pca.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(8, 6))
plt.scatter(reduced_features[:, 0], reduced_features[:, 1], alpha=0.7)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Projection Before Clustering")
plt.grid(alpha=0.2)
plt.show()

#### Evaluate multiple values of k using the Davies-Bouldin Index

K-Means is fitted for values of `k` from 2 to 20. A lower Davies-Bouldin Index indicates clusters that are more compact and better separated. The index is used as one source of evidence rather than an automatic final decision.

In [ ]:
# Evaluate K-Means solutions using the Davies-Bouldin Index
def calculate_davies_bouldin_scores(features, minimum_k=2, maximum_k=20):
    rows = []
    maximum_k = min(maximum_k, len(features) - 1)

    for k in range(minimum_k, maximum_k + 1):
        candidate_model = KMeans(
            n_clusters=k,
            n_init=10,
            random_state=RANDOM_STATE
        )
        candidate_labels = candidate_model.fit_predict(features)
        score = davies_bouldin_score(features, candidate_labels)
        rows.append({
            "Number of Clusters": k,
            "Davies-Bouldin Index": score
        })

    return pd.DataFrame(rows)


dbi_results = calculate_davies_bouldin_scores(reduced_features)

plt.figure(figsize=(10, 5))
plt.plot(
    dbi_results["Number of Clusters"],
    dbi_results["Davies-Bouldin Index"],
    marker="o"
)
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Davies-Bouldin Index")
plt.title("Davies-Bouldin Index for K-Means Solutions")
plt.xticks(dbi_results["Number of Clusters"])
plt.grid(alpha=0.3)
plt.show()

# Use k = 3, 5 and 6 as the candidate solutions for detailed review
candidate_values = [3, 5, 6]
top_candidates = (
    dbi_results[dbi_results["Number of Clusters"].isin(candidate_values)]
    .set_index("Number of Clusters")
    .loc[candidate_values]
    .reset_index()
)
print("Selected candidate values for detailed review:")
display(top_candidates)

#### Review the three candidate clustering solutions

Each candidate is assessed using three forms of evidence: its Davies-Bouldin Index, the separation visible in the first two principal components, and representative images closest to each cluster centroid. Cluster size is also reported to identify highly fragmented or imbalanced solutions.

In [ ]:
# Plot a two-dimensional PCA view of a clustering solution
def plot_pca_clusters(features, cluster_labels, k, title):
    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(
        features[:, 0],
        features[:, 1],
        c=cluster_labels,
        cmap="tab20",
        alpha=0.75
    )
    colour_bar = plt.colorbar(scatter, ticks=range(k))
    colour_bar.set_label("Cluster")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title(title)
    plt.grid(alpha=0.2)
    plt.show()


# Display images closest to each K-Means centroid
def show_representative_images(
    images,
    filenames,
    features,
    cluster_labels,
    cluster_centres,
    images_per_cluster=3,
    show_filenames=False,
    figure_title="Representative Images"
):
    number_of_clusters = len(cluster_centres)
    fig, axes = plt.subplots(
        number_of_clusters,
        images_per_cluster,
        figsize=(4 * images_per_cluster, 3.4 * number_of_clusters),
        squeeze=False
    )

    for cluster in range(number_of_clusters):
        cluster_indices = np.where(cluster_labels == cluster)[0]
        cluster_distances = np.linalg.norm(
            features[cluster_indices] - cluster_centres[cluster],
            axis=1
        )
        representative_indices = cluster_indices[np.argsort(cluster_distances)[:images_per_cluster]]

        for column in range(images_per_cluster):
            axis = axes[cluster, column]
            axis.axis("off")

            if column < len(representative_indices):
                image_index = representative_indices[column]
                axis.imshow(images[image_index])
                title = f"Cluster {cluster}"
                if show_filenames:
                    title += "\n" + textwrap.fill(str(filenames[image_index]), width=22)
                axis.set_title(title, fontsize=8 if show_filenames else 10)

    fig.suptitle(figure_title, fontsize=14, y=1.002)
    plt.tight_layout()
    plt.show()


In [ ]:
# Fit and inspect the three candidate solutions: k = 3, 5 and 6
candidate_models = {}

for candidate_k in candidate_values:
    candidate_model = KMeans(
        n_clusters=candidate_k,
        n_init=10,
        random_state=RANDOM_STATE
    )
    candidate_labels = candidate_model.fit_predict(reduced_features)
    candidate_models[candidate_k] = candidate_model

    candidate_dbi = davies_bouldin_score(reduced_features, candidate_labels)
    cluster_sizes = pd.Series(candidate_labels).value_counts().sort_index()

    print(f"\nCandidate k = {candidate_k}; Davies-Bouldin Index = {candidate_dbi:.4f}")
    display(
        cluster_sizes
        .rename_axis("Cluster")
        .to_frame("Number of Images")
    )

    plot_pca_clusters(
        reduced_features,
        candidate_labels,
        candidate_k,
        title=f"PCA Visualisation for Candidate k = {candidate_k}"
    )

    show_representative_images(
        images,
        filenames,
        reduced_features,
        candidate_labels,
        candidate_model.cluster_centers_,
        images_per_cluster=3,
        show_filenames=False,
        figure_title=f"Representative Images for Candidate k = {candidate_k}"
    )

#### Select and fit the final K-Means solution

Review the DBI values, PCA plots, cluster sizes and representative images above. Enter the final value of `k` based on the combined evidence rather than selecting the minimum DBI mechanically.

In [ ]:
# Set the final number of clusters after reviewing all candidate evidence
# Update this value after comparing DBI, PCA separation and representative images.
FINAL_K = 3

print(f"Candidate values selected for detailed review: {candidate_values}")
final_k = FINAL_K

maximum_evaluated_k = int(dbi_results["Number of Clusters"].max())
if final_k < 2 or final_k > maximum_evaluated_k:
    raise ValueError(f"final_k must be between 2 and {maximum_evaluated_k}.")

final_kmeans = KMeans(
    n_clusters=final_k,
    n_init=10,
    random_state=RANDOM_STATE
)
final_labels = final_kmeans.fit_predict(reduced_features)
final_dbi = davies_bouldin_score(reduced_features, final_labels)

print(f"Final k: {final_k}")
print(f"Final Davies-Bouldin Index: {final_dbi:.4f}")

plot_pca_clusters(
    reduced_features,
    final_labels,
    final_k,
    title=f"Final K-Means Clusters (k = {final_k})"
)

#### Inspect the final clusters

The table reports the number of images in each cluster. The displayed images are the observations closest to each cluster centroid and therefore provide representative examples for manual assessment. Filenames are included to support follow-up inspection of the original files.

In [ ]:
# Report final cluster sizes
final_cluster_sizes = (
    pd.Series(final_labels)
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .to_frame("Number of Images")
)
display(final_cluster_sizes)

# Display representative images and filenames for manual review
show_representative_images(
    images,
    filenames,
    reduced_features,
    final_labels,
    final_kmeans.cluster_centers_,
    images_per_cluster=6,
    show_filenames=True,
    figure_title=f"Final Cluster Representatives (k = {final_k})"
)

#### Manually assess the visual characteristics of each cluster

After reviewing the representative images, assign one of three assessments to each cluster. The reviewed values are saved in the notebook so the clustering export can be reproduced without interactive input:

* `damaged` - the cluster predominantly displays visible damage.
* `undamaged` - the cluster predominantly displays clothes without clear damage.
* `unclear` - the cluster is visually mixed or there is insufficient evidence for a reliable assessment.

In [ ]:
# Record a manual assessment for each final cluster.
# Update these values after reviewing the final representative images.
cluster_assessments = {
    0: "unclear",
    1: "unclear",
    2: "unclear",
}

valid_assessments = {"damaged", "undamaged", "unclear"}
expected_clusters = set(range(final_k))
provided_clusters = set(cluster_assessments.keys())

if provided_clusters != expected_clusters:
    raise ValueError(
        f"cluster_assessments must contain exactly these clusters: {sorted(expected_clusters)}"
    )

invalid_assessments = set(cluster_assessments.values()) - valid_assessments
if invalid_assessments:
    raise ValueError(
        f"Invalid cluster assessments found: {sorted(invalid_assessments)}"
    )

cluster_assessments

#### Summarise and export the clustering results

The final summary combines cluster size and manual assessment. `outputs/clustering_results.csv` records the cluster assigned to every image and the corresponding cluster-level assessment, providing a foundation for later manual review or semi-supervised labelling.

In [ ]:
# Create the cluster summary
cluster_summary = final_cluster_sizes.reset_index()
cluster_summary["Cluster_Assessment"] = cluster_summary["Cluster"].map(cluster_assessments)
display(cluster_summary)

# Export one row for every image
clustering_results = pd.DataFrame({
    "Filename": filenames,
    "Cluster": final_labels
})
clustering_results["Cluster_Assessment"] = (
    clustering_results["Cluster"].map(cluster_assessments)
)
clustering_results.to_csv(OUTPUT_DIR / "clustering_results.csv", index=False)

print("outputs/clustering_results.csv has been created successfully.")
display(clustering_results.head(10))